# WaterPulse — RAG Pipeline Notebook
**Purpose:** Build a FAISS vector index over MIS 78th Round water data so that
JalMitra can retrieve factual context before generating answers with IBM Granite.

## Architecture
```
Db2 STATE_WATER_METRICS
      │
      ▼
  Text Chunks  ──► watsonx.ai Embeddings (ibm/slate-125m-english-rtrvr)
      │
      ▼
  FAISS Index  ──► saved to COS (waterpulse-raw-data/rag/faiss_index.bin)
      │
      ▼  (at query time)
  Top-k chunks ──► IBM Granite (ibm-granite-3-2-8b-instruct via watsonx.ai)
      │
      ▼
  Grounded Answer (JalMitra response)
```

## Steps
1. Load cleaned data from Db2
2. Convert each state row into a rich natural-language text chunk
3. Embed chunks using watsonx.ai Slate embedding model
4. Build FAISS index and save to COS
5. Test end-to-end RAG query with IBM Granite


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install',
    'ibm-watsonx-ai>=0.2.6',
    'faiss-cpu>=1.7.4',
    'ibm-db',
    'ibm-cos-sdk',
    'numpy',
    'pandas'
], check=True)
print('Dependencies installed')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, io, json, pickle
import numpy as np
import pandas as pd
import faiss
import ibm_db
import ibm_boto3
from ibm_botocore.client import Config
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.foundation_models.utils.enums import EmbeddingTypes, ModelTypes
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

print('All imports successful')

In [ ]:
# ── Credentials — replace with your actual values ────────────────────────────

# IBM watsonx.ai (Watson Machine Learning → now watsonx.ai)
WATSONX_API_KEY  = 'YOUR_WATSONX_API_KEY'
WATSONX_URL      = 'https://us-south.ml.cloud.ibm.com'
WATSONX_PROJECT_ID = 'YOUR_WATSONX_PROJECT_ID'

# IBM Cloud Object Storage
COS_API_KEY      = 'YOUR_COS_API_KEY'
COS_INSTANCE_CRN = 'YOUR_COS_INSTANCE_CRN'
COS_ENDPOINT     = 'https://s3.us-south.cloud-object-storage.appdomain.cloud'
COS_BUCKET       = 'waterpulse-raw-data'

# IBM Db2 on Cloud
DB2_DSN = (
    'DATABASE=BLUDB;'
    'HOSTNAME=YOUR_DB2_HOST.databases.appdomain.cloud;'
    'PORT=30376;PROTOCOL=TCPIP;'
    'UID=YOUR_USERNAME;PWD=YOUR_PASSWORD;SECURITY=SSL;'
)

print('Credentials configured')

In [ ]:
# ── Step 1: Load STATE_WATER_METRICS from Db2 ────────────────────────────────
conn = ibm_db.connect(DB2_DSN, '', '')

sql = """
    SELECT state_code, state_name, year,
           rural_pct, urban_pct, total_pct,
           equity_gap, sdg61_proxy, yoy_change, sdg_risk_flag,
           cook_rural_pct, cook_urban_pct,
           migration_rate, net_migration_000s
    FROM WATERPULSE.STATE_WATER_METRICS
    ORDER BY state_name
"""

stmt = ibm_db.exec_immediate(conn, sql)
rows = []
row = ibm_db.fetch_assoc(stmt)
while row:
    rows.append(row)
    row = ibm_db.fetch_assoc(stmt)
ibm_db.close(conn)

df = pd.DataFrame(rows)
# Normalize column names to lowercase
df.columns = [c.lower() for c in df.columns]
print(f'Loaded {len(df)} rows from Db2')
df.head()

In [ ]:
# ── Step 2: Convert each state row into a rich text chunk ────────────────────
# Each chunk is a natural-language paragraph describing one state's water situation.
# This is what Granite will receive as context when answering user questions.

def row_to_chunk(row):
    sdg_status = 'On Track' if float(row.get('sdg_risk_flag', 1)) == 0 else 'At Risk'
    tier_map = {
        (True,  False): 'Tier A (High Access, Low Disparity)',
        (True,  True):  'Tier B (High Access, High Disparity)',
        (False, False): 'Tier C (Low Access, Low Disparity)',
        (False, True):  'Tier D (Low Access, High Disparity — Priority Intervention)'
    }
    high_access  = float(row.get('sdg61_proxy', 0)) >= 80
    high_dispar  = float(row.get('equity_gap',  0)) >= 15
    tier         = tier_map[(high_access, high_dispar)]

    yoy = row.get('yoy_change', 0)
    yoy_text = f"improved by {abs(float(yoy)):.1f} percentage points" if float(yoy or 0) > 0 \
               else f"declined by {abs(float(yoy)):.1f} percentage points" if float(yoy or 0) < 0 \
               else "remained stable"

    chunk = (
        f"State: {row['state_name']} (State Code: {row['state_code']}, Year: {row['year']})\n"
        f"Improved drinking water source access — "
        f"Rural: {row['rural_pct']}%, Urban: {row['urban_pct']}%, Total: {row['total_pct']}%.\n"
        f"Rural-urban equity gap: {row['equity_gap']} percentage points "
        f"({'critical' if float(row['equity_gap']) > 20 else 'high' if float(row['equity_gap']) > 15 else 'moderate'}).\n"
        f"SDG 6.1 proxy score: {row['sdg61_proxy']}% — SDG Status: {sdg_status}.\n"
        f"Year-over-year change: Total access {yoy_text} compared to previous year.\n"
        f"Clean cooking fuel access — Rural: {row['cook_rural_pct']}%, Urban: {row['cook_urban_pct']}%.\n"
        f"Migration rate: {row['migration_rate']}% "
        f"(Net migration: {row['net_migration_000s']} thousand).\n"
        f"ML Equity Tier Classification: {tier}.\n"
        f"Policy recommendation: "
        + (
            "Urgent intervention required. Prioritize JJM fund allocation, rural piped water expansion, "
            "and bundled SDG 6+7 programs for cooking fuel and water together."
            if not high_access and high_dispar
            else "Continue current programs. Monitor equity gap to prevent worsening."
            if high_access and high_dispar
            else "On track but scale-up needed to close remaining gaps by 2030."
        )
    )
    return chunk

chunks    = [row_to_chunk(row) for _, row in df.iterrows()]
chunk_meta = [{'state': row['state_name'], 'state_code': row['state_code'],
               'year': row['year']} for _, row in df.iterrows()]

print(f'Generated {len(chunks)} text chunks')
print('\n── Sample chunk (Bihar) ──')
bihar_idx = next(i for i,r in enumerate(chunk_meta) if 'Bihar' in str(r.get('state','')))
print(chunks[bihar_idx])

In [ ]:
# ── Step 3: Embed chunks using watsonx.ai Slate embedding model ──────────────
# Model: ibm/slate-125m-english-rtrvr — available on watsonx.ai Lite

credentials = Credentials(
    url=WATSONX_URL,
    api_key=WATSONX_API_KEY
)

embedding_model = ModelInference(
    model_id=EmbeddingTypes.IBM_SLATE_125M_ENG,
    credentials=credentials,
    project_id=WATSONX_PROJECT_ID
)

# Embed in batches of 10 to respect rate limits
BATCH_SIZE = 10
all_embeddings = []

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    resp  = embedding_model.embed_documents(texts=batch)
    vecs  = [item['embedding'] for item in resp['results']]
    all_embeddings.extend(vecs)
    print(f'Embedded batch {i//BATCH_SIZE + 1}/{(len(chunks)-1)//BATCH_SIZE + 1} '
          f'({len(all_embeddings)}/{len(chunks)} chunks)')

embeddings_np = np.array(all_embeddings, dtype='float32')
print(f'\nEmbedding matrix shape: {embeddings_np.shape}')
print(f'Embedding dimension: {embeddings_np.shape[1]}')

In [ ]:
# ── Step 4: Build FAISS index ─────────────────────────────────────────────────
dim   = embeddings_np.shape[1]
index = faiss.IndexFlatIP(dim)          # Inner Product = cosine similarity (after L2 norm)

# Normalize for cosine similarity
faiss.normalize_L2(embeddings_np)
index.add(embeddings_np)

print(f'FAISS index built: {index.ntotal} vectors, dim={dim}')

# Verify with a test query
test_query_embedding = embedding_model.embed_query(text='rural water access Bihar equity gap')
test_vec = np.array([test_query_embedding], dtype='float32')
faiss.normalize_L2(test_vec)

D, I = index.search(test_vec, k=3)
print('\n── Top-3 retrieval test for "rural water access Bihar equity gap" ──')
for rank, (dist, idx) in enumerate(zip(D[0], I[0])):
    print(f'{rank+1}. {chunk_meta[idx]["state"]} (score={dist:.4f})')

In [ ]:
# ── Step 5: Serialize and save FAISS index + metadata to COS ─────────────────
cos_client = ibm_boto3.client(
    service_name='s3',
    ibm_api_key_id=COS_API_KEY,
    ibm_service_instance_id=COS_INSTANCE_CRN,
    config=Config(signature_version='oauth'),
    endpoint_url=COS_ENDPOINT
)

# Save FAISS index binary
faiss_buf = io.BytesIO()
faiss.write_index(index, '/tmp/waterpulse_faiss.index')
with open('/tmp/waterpulse_faiss.index', 'rb') as f:
    cos_client.put_object(Bucket=COS_BUCKET, Key='rag/faiss_index.bin', Body=f.read())

# Save chunks + metadata as JSON (needed at query time to return text to Granite)
rag_store = {
    'chunks':     chunks,
    'metadata':   chunk_meta,
    'embed_dim':  dim,
    'model':      'ibm/slate-125m-english-rtrvr',
    'built_at':   pd.Timestamp.now().isoformat()
}
cos_client.put_object(
    Bucket=COS_BUCKET,
    Key='rag/chunk_store.json',
    Body=json.dumps(rag_store, ensure_ascii=False)
)

print('Saved to COS:')
print(f'  rag/faiss_index.bin   ({index.ntotal} vectors)')
print(f'  rag/chunk_store.json  ({len(chunks)} chunks)')

In [ ]:
# ── Step 6: End-to-end RAG test with IBM Granite ──────────────────────────────
# This is exactly what Cloud Functions getNLQuery() will do at runtime.

def rag_query(question: str, top_k: int = 3) -> str:
    """Full RAG pipeline: embed → retrieve → Granite generate."""

    # 1. Embed the question
    q_emb = embedding_model.embed_query(text=question)
    q_vec = np.array([q_emb], dtype='float32')
    faiss.normalize_L2(q_vec)

    # 2. Retrieve top-k most relevant state chunks
    D, I = index.search(q_vec, k=top_k)
    retrieved = [chunks[i] for i in I[0]]
    context   = '\n\n---\n\n'.join(retrieved)

    # 3. Build prompt for IBM Granite
    prompt = f"""You are JalMitra, an AI assistant specializing in India's drinking water access data
from the MIS 78th Round survey (2020-21). Answer questions accurately using only the
provided context. Be concise, cite specific numbers, and focus on policy implications.

Context (retrieved MIS 78th Round data):
{context}

Question: {question}

Answer:"""

    # 4. Generate answer with IBM Granite (ibm-granite-3-2-8b-instruct)
    granite = ModelInference(
        model_id='ibm/granite-3-2-8b-instruct',
        credentials=credentials,
        project_id=WATSONX_PROJECT_ID,
        params={
            GenParams.MAX_NEW_TOKENS: 512,
            GenParams.TEMPERATURE:    0.1,      # low temperature for factual answers
            GenParams.TOP_P:          0.9,
            GenParams.REPETITION_PENALTY: 1.1,
            GenParams.STOP_SEQUENCES: ['\n\nQuestion:', '\n\nContext:']
        }
    )

    response = granite.generate_text(prompt=prompt)
    return response

# ── Test 1: State-specific query ──
print('=== Test 1: Bihar water access ===')
ans = rag_query('What is the rural water access situation in Bihar and is it on track for SDG 6?')
print(ans)

print('\n=== Test 2: Priority states ===')
ans2 = rag_query('Which states require urgent intervention for drinking water access?')
print(ans2)

print('\n=== Test 3: Cooking fuel correlation ===')
ans3 = rag_query('How does clean cooking fuel access relate to rural water access?')
print(ans3)

In [ ]:
# ── Step 7: Save RAG query config for Cloud Functions ────────────────────────
# Cloud Functions will load the FAISS index from COS at cold-start
# and use these parameters for every getNLQuery() call.

rag_config = {
    'watsonx_url':         WATSONX_URL,
    'embedding_model_id':  'ibm/slate-125m-english-rtrvr',
    'granite_model_id':    'ibm/granite-3-2-8b-instruct',
    'faiss_index_cos_key': 'rag/faiss_index.bin',
    'chunk_store_cos_key': 'rag/chunk_store.json',
    'top_k':               3,
    'max_new_tokens':      512,
    'temperature':         0.1
}

cos_client.put_object(
    Bucket=COS_BUCKET,
    Key='rag/rag_config.json',
    Body=json.dumps(rag_config, indent=2)
)

print('RAG config saved to COS: rag/rag_config.json')
print('\n── RAG Pipeline Build Complete ──')
print(f'  Chunks indexed:      {len(chunks)}')
print(f'  Embedding model:     ibm/slate-125m-english-rtrvr')
print(f'  Generation model:    ibm/granite-3-2-8b-instruct')
print(f'  FAISS index dim:     {dim}')
print(f'  Index stored at:     cos://{COS_BUCKET}/rag/faiss_index.bin')